# 01 – Khám phá dữ liệu (EDA) & Pipeline dữ liệu

Dữ liệu: **UIT-VSFC** (phản hồi sinh viên), 3 nhãn `0 = Tiêu cực`, `1 = Trung tính`, `2 = Tích cực`.

Notebook này chỉ **gọi lại code trong `src/`** – logic không bị nhân bản giữa các notebook.

In [ ]:
# Cho phép import src/ dù mở notebook từ thư mục notebooks/
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config, resolve
cfg = load_config()
print("Project root:", ROOT)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from src.preprocessing import load_split

raw = {s: load_split(cfg, s) for s in ("train", "validation", "test")}
summary = pd.DataFrame({s: d["label"].value_counts().sort_index() for s, d in raw.items()})
summary.index = [cfg["labels"][i] for i in summary.index]
summary.loc["Tổng"] = summary.sum()
summary

In [ ]:
fig_dir = resolve(cfg["results"]["figures_dir"]); fig_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=False)
for ax, (s, d) in zip(axes, raw.items()):
    d["label"].map(cfg["labels"]).value_counts().reindex(cfg["labels"].values()).plot.bar(ax=ax, color=["#d9534f", "#f0ad4e", "#5cb85c"])
    ax.set_title(f"{s} (n={len(d):,})"); ax.tick_params(axis="x", rotation=0)
fig.suptitle("Phân bố nhãn – lớp Trung tính chỉ ~4%  → mất cân bằng nặng")
fig.tight_layout(); fig.savefig(fig_dir / "label_dist.png", dpi=150); plt.show()

In [ ]:
raw["train"]["n_words"] = raw["train"]["text"].str.split().str.len()
print(raw["train"].groupby("label")["n_words"].describe().round(1))
ax = raw["train"]["n_words"].clip(upper=60).hist(bins=60, figsize=(8, 3.5))
ax.set_xlabel("Số từ / câu (cắt ở 60)"); ax.set_ylabel("Số câu"); ax.set_title("Độ dài câu tập train")
plt.tight_layout(); plt.savefig(fig_dir / "word_count.png", dpi=150); plt.show()

## Tiền xử lý
Cùng **một** hàm `preprocess()` cho train / validation / test / demo → tránh lệch phân phối lúc suy luận.

In [ ]:
from src.preprocessing import clean_text, prepare_data

for s in ["GV dạy rất HAY!!!", "ko hiểu bài", "nội dung cô đọng", "thầy dạy tạm ổn 😍"]:
    print(f"{s!r:30} -> {clean_text(s)!r}")

In [ ]:
splits = prepare_data(cfg)   # ghi data/processed/{train,validation,test}.csv
splits["train"].head()

## Tokenizer & DataLoader
Cần internet lần đầu để tải `vinai/phobert-base`.

In [ ]:
from transformers import AutoTokenizer
from src.dataset import build_loader

tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["pretrained_name"])
lens = splits["train"]["text_clean"].map(lambda t: len(tokenizer(t)["input_ids"]))
print(lens.describe(percentiles=[.5, .95, .99]).round(1))
print(f"Tỉ lệ câu dài hơn max_length={cfg['model']['max_length']}: {(lens > cfg['model']['max_length']).mean():.3%}")

In [ ]:
loader = build_loader(splits["train"], tokenizer, cfg, shuffle=True, batch_size=cfg["training"]["batch_size"])
batch = next(iter(loader))
for k, v in batch.items():
    print(f"{k:15s} {tuple(v.shape)} {v.dtype}")
# dynamic padding: độ dài mỗi batch = câu dài nhất trong batch, không phải luôn 256